# 04 — Stacked Ensemble

Combines LightGBM and MLP predictions using a Logistic Regression meta-learner.

Why stacking works here:
- LightGBM excels at categorical interactions and handles missing values natively
- MLP captures smooth non-linear patterns LightGBM misses
- LR meta-learner learns optimal weighting between them on val data

**Inputs:** `*_train_preds.npy` and `*_val_preds.npy` from notebook 03  
**Output:** `meta_learner.pkl` — the final production model

In [ ]:
import numpy as np
import pandas as pd
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

# Load base model predictions
lgbm_train = np.load('/kaggle/working/lgbm_train_preds.npy')
mlp_train  = np.load('/kaggle/working/mlp_train_preds.npy')
lgbm_val   = np.load('/kaggle/working/lgbm_val_preds.npy')
mlp_val    = np.load('/kaggle/working/mlp_val_preds.npy')

train = pd.read_parquet('/kaggle/working/train_features.parquet')
val   = pd.read_parquet('/kaggle/working/val_features.parquet')

y_train = train['isFraud'].values
y_val   = val['isFraud'].values

# Stack predictions
X_meta_train = np.column_stack([lgbm_train, mlp_train])
X_meta_val   = np.column_stack([lgbm_val, mlp_val])

print(f'Meta-learner input shape: {X_meta_train.shape}')
print(f'Base LightGBM Val PR-AUC: {average_precision_score(y_val, lgbm_val):.4f}')
print(f'Base MLP Val PR-AUC:      {average_precision_score(y_val, mlp_val):.4f}')

In [ ]:
# Meta-learner: Logistic Regression on val predictions
# Trained on VAL so it hasn't seen these examples during base model training
meta_learner = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
meta_learner.fit(X_meta_train, y_train)

ensemble_val_preds = meta_learner.predict_proba(X_meta_val)[:, 1]

print(f'Ensemble Val ROC-AUC: {roc_auc_score(y_val, ensemble_val_preds):.4f}')
print(f'Ensemble Val PR-AUC:  {average_precision_score(y_val, ensemble_val_preds):.4f}')
print(f'\nMeta-learner weights: LightGBM={meta_learner.coef_[0][0]:.3f}, MLP={meta_learner.coef_[0][1]:.3f}')

np.save('/kaggle/working/ensemble_val_preds.npy', ensemble_val_preds)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, preds in [('LightGBM', lgbm_val), ('MLP', mlp_val), ('Ensemble', ensemble_val_preds)]:
    prec, rec, _ = precision_recall_curve(y_val, preds)
    pr_auc = average_precision_score(y_val, preds)
    axes[0].plot(rec, prec, label=f'{name} (PR-AUC={pr_auc:.3f})')

axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve (Validation Set)')
axes[0].legend()
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1])

# Threshold sensitivity
thresholds = np.linspace(0.1, 0.9, 80)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    preds_bin = (ensemble_val_preds >= t).astype(int)
    tp = ((preds_bin == 1) & (y_val == 1)).sum()
    fp = ((preds_bin == 1) & (y_val == 0)).sum()
    fn = ((preds_bin == 0) & (y_val == 1)).sum()
    p = tp / max(tp + fp, 1)
    r = tp / max(tp + fn, 1)
    f = 2 * p * r / max(p + r, 1e-9)
    precisions.append(p)
    recalls.append(r)
    f1s.append(f)

axes[1].plot(thresholds, precisions, label='Precision')
axes[1].plot(thresholds, recalls, label='Recall')
axes[1].plot(thresholds, f1s, label='F1', linewidth=2)
axes[1].axvline(x=0.70, color='crimson', linestyle='--', label='Block threshold (0.70)')
axes[1].axvline(x=0.35, color='orange', linestyle='--', label='Review threshold (0.35)')
axes[1].set_xlabel('Threshold')
axes[1].set_title('Threshold Sensitivity')
axes[1].legend()

plt.tight_layout()
plt.savefig('/kaggle/working/pr_curve_and_threshold.png', dpi=150)
plt.show()

In [ ]:
# Save meta-learner
with open('/kaggle/working/meta_learner.pkl', 'wb') as f:
    pickle.dump(meta_learner, f)

print('Meta-learner saved. Ready for 05_evaluation.ipynb')